# 027 — Self-Querying RAG
**Advanced RAG Series** | LangChain SelfQueryRetriever: NL query → semantic search + metadata filter

Covers: Metadata-enriched indexing · AttributeInfo · SelfQueryRetriever · Filter + query separation


In [1]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


✓ API key loaded


In [2]:
import os
os.environ["ANONYMIZED_TELEMETRY"] = "False"

from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")


✓ llm       : llama-3.1-8b-instant
✓ llm_smart : llama-3.3-70b-versatile


In [3]:
import os
DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    return open(path, encoding='utf-8').read()

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
rag_text = load_doc("rag.txt")
dl_text  = load_doc("deep_learning.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, rag_text])
print(f"✓ Loaded {len(all_text):,} chars")


✓ Loaded 22,486 chars


In [4]:
# ── 1. The problem: queries mix semantic and structured intent ───────────
print("Problem with standard RAG retrieval:")
print()
print("  Query: 'papers about transformers published after 2020'")
print("  Standard RAG: embeds the whole query as one vector.")
print("  It cannot filter by 'published after 2020' — that's a structured constraint.")
print()
print("  Query: 'deep learning techniques for NLP, not from Wikipedia'")
print("  Standard RAG: no way to exclude by source.")
print()
print("Self-Querying RAG:")
print("  LLM decomposes the query into:")
print("    1. Semantic part  : 'transformers NLP' (for vector similarity)")
print("    2. Structured part: year > 2020 AND source != Wikipedia (metadata filter)")
print("  Apply structured filter FIRST, then semantic search on the filtered set.")


Problem with standard RAG retrieval:

  Query: 'papers about transformers published after 2020'
  Standard RAG: embeds the whole query as one vector.
  It cannot filter by 'published after 2020' — that's a structured constraint.

  Query: 'deep learning techniques for NLP, not from Wikipedia'
  Standard RAG: no way to exclude by source.

Self-Querying RAG:
  LLM decomposes the query into:
    1. Semantic part  : 'transformers NLP' (for vector similarity)
    2. Structured part: year > 2020 AND source != Wikipedia (metadata filter)
  Apply structured filter FIRST, then semantic search on the filtered set.


In [5]:
# ── 2. Build a Chroma vector store with rich metadata ───────────────────
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

os.environ["ANONYMIZED_TELEMETRY"] = "False"

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    cache_folder=os.path.join(os.getcwd(), '..', 'datasets', 'models'),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# Build documents with meaningful metadata
source_meta = [
    ("ai.txt",              {"source": "ai_overview",    "topic": "AI",            "year": 2023, "level": "intro"}),
    ("machine_learning.txt",{"source": "ml_guide",       "topic": "machine_learning","year": 2022,"level": "intermediate"}),
    ("nlp.txt",             {"source": "nlp_reference",  "topic": "NLP",           "year": 2023, "level": "intermediate"}),
    ("rag.txt",             {"source": "rag_manual",     "topic": "RAG",           "year": 2024, "level": "advanced"}),
    ("deep_learning.txt",   {"source": "dl_textbook",    "topic": "deep_learning", "year": 2021, "level": "advanced"}),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=350, chunk_overlap=35)
all_docs = []
DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')
for fname, meta in source_meta:
    text = open(os.path.join(DS_DIR, fname), encoding='utf-8').read()
    chunks = splitter.split_text(text)
    for chunk in chunks:
        all_docs.append(Document(page_content=chunk, metadata=dict(meta)))

print(f"✓ {len(all_docs)} documents with metadata")
print(f"  Sample metadata: {all_docs[0].metadata}")

# Build Chroma store
vectorstore = Chroma.from_documents(
    all_docs, embeddings,
    collection_name="self_query_demo",
)
print(f"✓ Chroma store: {vectorstore._collection.count()} vectors")


✓ 118 documents with metadata
  Sample metadata: {'source': 'ai_overview', 'topic': 'AI', 'year': 2023, 'level': 'intro'}


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✓ Chroma store: 118 vectors


In [6]:
# ── 3. Define metadata schema for SelfQueryRetriever ────────────────────
from langchain.chains.query_constructor.base import AttributeInfo
from langchain.retrievers.self_query.base import SelfQueryRetriever

metadata_field_info = [
    AttributeInfo(
        name="topic",
        description="The main subject area of the document. "
                    "One of: AI, machine_learning, NLP, RAG, deep_learning",
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="The year the document was written or last updated.",
        type="integer",
    ),
    AttributeInfo(
        name="level",
        description="Difficulty level of the content. "
                    "One of: intro, intermediate, advanced",
        type="string",
    ),
    AttributeInfo(
        name="source",
        description="The source name of the document.",
        type="string",
    ),
]

document_content_description = "Technical documents about AI, ML, NLP, deep learning, and RAG."

retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    verbose=True,
    search_kwargs={"k": 4},
)
print("✓ SelfQueryRetriever ready")


✓ SelfQueryRetriever ready


In [7]:
# ── 4. Demo: structured filter + semantic query ──────────────────────────
# Query 1: filter by topic + semantic
print("Query 1: 'How does retrieval work in RAG systems?'")
print("-" * 50)
docs1 = retriever.invoke("How does retrieval work in RAG systems?")
for d in docs1[:2]:
    print(f"  [{d.metadata}]")
    print(f"  {d.page_content[:150]}")
    print()


Query 1: 'How does retrieval work in RAG systems?'
--------------------------------------------------


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


  [{'level': 'advanced', 'source': 'rag_manual', 'topic': 'RAG', 'year': 2024}]
  RAG Evaluation

RAGAS (Retrieval-Augmented Generation Assessment) provides automated metrics:
- Faithfulness: is the answer supported by the retrieved

  [{'level': 'advanced', 'source': 'rag_manual', 'topic': 'RAG', 'year': 2024}]
  Retrieval-Augmented Generation (RAG)

Retrieval-Augmented Generation (RAG) is a hybrid AI architecture that combines information
retrieval with text g



In [8]:
# Query 2: filter by year
print("Query 2: 'advanced techniques from 2023 or later'")
print("-" * 50)
docs2 = retriever.invoke("advanced techniques from 2023 or later")
for d in docs2[:2]:
    print(f"  [{d.metadata}]")
    print(f"  {d.page_content[:150]}")
    print()


Query 2: 'advanced techniques from 2023 or later'
--------------------------------------------------


  [{'level': 'advanced', 'source': 'rag_manual', 'topic': 'RAG', 'year': 2024}]
  Vector Databases

  [{'level': 'advanced', 'source': 'rag_manual', 'topic': 'RAG', 'year': 2024}]
  1. Ingestion: Documents are loaded, split into chunks, embedded into vectors, and stored in
   a vector database. The chunk size and overlap are criti



In [9]:
# Query 3: filter by level
print("Query 3: 'intro level explanation of neural networks'")
print("-" * 50)
docs3 = retriever.invoke("intro level explanation of neural networks")
for d in docs3[:2]:
    print(f"  [{d.metadata}]")
    print(f"  {d.page_content[:150]}")
    print()


Query 3: 'intro level explanation of neural networks'
--------------------------------------------------


  [{'level': 'intro', 'source': 'ai_overview', 'topic': 'AI', 'year': 2023}]
  Artificial Intelligence

  [{'level': 'intro', 'source': 'ai_overview', 'topic': 'AI', 'year': 2023}]
  An artificial neural network (ANN) is modeled loosely after the human brain that is designed to
recognize patterns. They interpret sensory data throug



In [10]:
# ── 5. Manual structured query to see what LLM generates ────────────────
from langchain.chains.query_constructor.base import (
    StructuredQueryOutputParser, get_query_constructor_prompt
)

prompt = get_query_constructor_prompt(
    document_content_description,
    metadata_field_info,
)
output_parser = StructuredQueryOutputParser.from_components()
query_constructor = prompt | llm | output_parser

test_query = "NLP documents at intermediate or advanced level from 2022 or later"
structured = query_constructor.invoke({"query": test_query})
print(f"Input query: {test_query!r}")
print(f"\nStructured query generated:")
print(f"  Semantic part : {structured.query!r}")
print(f"  Filter        : {structured.filter}")


Input query: 'NLP documents at intermediate or advanced level from 2022 or later'

Structured query generated:
  Semantic part : 'NLP'
  Filter        : operator=<Operator.AND: 'and'> arguments=[Operation(operator=<Operator.OR: 'or'>, arguments=[Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='level', value='intermediate'), Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='level', value='advanced')]), Comparison(comparator=<Comparator.GTE: 'gte'>, attribute='year', value=2022)]


In [11]:
# ── 6. Full self-querying RAG pipeline ──────────────────────────────────
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = ChatPromptTemplate.from_template(
    "Answer ONLY from the context below. If not found, say 'I don't know'.\n\n"
    "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
)

def self_query_rag(question: str) -> str:
    docs = retriever.invoke(question)
    context = "\n\n---\n\n".join([
        f"[{d.metadata['topic']}, {d.metadata['year']}]\n{d.page_content}"
        for d in docs
    ])
    return (RAG_PROMPT | llm).invoke({"context": context, "question": question}).content

answer = self_query_rag("Explain attention mechanism from advanced NLP resources")
print(answer)


I don't know


In [12]:
# ── 7. Self-Querying vs Standard RAG ────────────────────────────────────
print("SELF-QUERYING RAG vs STANDARD RAG")
print("=" * 45)
print()
print("Standard RAG:")
print("  Embeds full query as one vector → finds similar chunks")
print("  Cannot enforce structured constraints (date, category, source)")
print()
print("Self-Querying RAG:")
print("  LLM decomposes query → semantic part + metadata filter")
print("  Applies Chroma/Pinecone metadata filters before similarity search")
print("  Result: only relevant documents by metadata ARE searched")
print()
print("When to use:")
print("  Your corpus has meaningful metadata (dates, authors, categories)")
print("  Users ask structured questions: 'show me X from year Y about Z'")
print("  You want to reduce retrieval noise by pre-filtering")
print()
print("Supported filter ops:")
print("  eq, ne, gt, gte, lt, lte  (for int/float/string)")
print("  and, or, not")


SELF-QUERYING RAG vs STANDARD RAG

Standard RAG:
  Embeds full query as one vector → finds similar chunks
  Cannot enforce structured constraints (date, category, source)

Self-Querying RAG:
  LLM decomposes query → semantic part + metadata filter
  Applies Chroma/Pinecone metadata filters before similarity search
  Result: only relevant documents by metadata ARE searched

When to use:
  Your corpus has meaningful metadata (dates, authors, categories)
  Users ask structured questions: 'show me X from year Y about Z'
  You want to reduce retrieval noise by pre-filtering

Supported filter ops:
  eq, ne, gt, gte, lt, lte  (for int/float/string)
  and, or, not
